# Energy Price Prediction with Machine Learning

## Objective
Predict hourly electricity prices (`price_eur_mwh`) using:
* Weather features (temperature, wind speed, pressure)
* Load features (actual load, forecast)
* Generation features (renewable/non-renewable shares)
* Temporal features (hour, day of week, month)

## Dataset
* **Source**: `dbacademy.labuser13955035_1772775399.h2_gold_model_scoring_base`
* **Records**: 17,535 hourly observations (2020-2021)
* **Zone**: Netherlands (NL)

## Models
* Random Forest Regressor
* XGBoost Regressor
* MLflow tracking for experiment management

In [0]:
# Configuration
import warnings
warnings.filterwarnings('ignore')

catalog = "dbacademy"
schema = "labuser13955035_1772775399"
source_table = f"{catalog}.{schema}.h2_gold_model_scoring_base"

print(f"Source Table: {source_table}")
print(f"\nML Configuration:")
print(f"  Target Variable: price_eur_mwh")
print(f"  Test Size: 20%")
print(f"  Random State: 42")

In [0]:
# Load ML-ready dataset
from pyspark.sql import functions as F

df = spark.table(source_table)

print(f"Dataset Shape: {df.count():,} rows, {len(df.columns)} columns")
print(f"\nColumns:")
for col in df.columns:
    print(f"  - {col}")

# Show sample
display(df.limit(10))

In [0]:
# Check for nulls and basic statistics
print("NULL VALUE CHECK")
print("="*80)

for col in df.columns:
    null_count = df.filter(F.col(col).isNull()).count()
    if null_count > 0:
        print(f"  ⚠️  {col}: {null_count:,} nulls")
    else:
        print(f"  ✅ {col}: no nulls")

# Summary statistics for target variable
print(f"\n\nTARGET VARIABLE: price_eur_mwh")
print("="*80)
stats = df.select(
    F.min("price_eur_mwh").alias("min"),
    F.max("price_eur_mwh").alias("max"),
    F.mean("price_eur_mwh").alias("mean"),
    F.stddev("price_eur_mwh").alias("std")
).first()

print(f"  Min:  {stats['min']:.2f} EUR/MWh")
print(f"  Max:  {stats['max']:.2f} EUR/MWh")
print(f"  Mean: {stats['mean']:.2f} EUR/MWh")
print(f"  Std:  {stats['std']:.2f} EUR/MWh")

In [0]:
# Prepare features for ML
import pandas as pd
import numpy as np

# Convert to Pandas for sklearn
pdf = df.toPandas()

# Define feature columns
feature_cols = [
    'avg_actual_total_load_mw',
    'day_ahead_total_load_forecast_mw',
    'temperature_c',
    'u10_ms',
    'v10_ms',
    'wind_speed_ms',
    'surface_pressure_pa',
    'renewable_generation_mw',
    'non_renewable_generation_mw',
    'renewable_share_pct',
    'non_renewable_share_pct',
    'hour_of_day',
    'day_of_week',
    'month'
]

target_col = 'price_eur_mwh'

# Remove rows with nulls
pdf_clean = pdf.dropna(subset=feature_cols + [target_col])

print(f"Original rows: {len(pdf):,}")
print(f"After removing nulls: {len(pdf_clean):,}")
print(f"\nFeatures: {len(feature_cols)}")
print(f"Target: {target_col}")

In [0]:
from sklearn.model_selection import train_test_split

# Split features and target
X = pdf_clean[feature_cols]
y = pdf_clean[target_col]

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print(f"Training Set: {len(X_train):,} samples")
print(f"Test Set: {len(X_test):,} samples")
print(f"\nFeature Matrix Shape: {X_train.shape}")
print(f"\nTarget Statistics (Train):")
print(f"  Min:  {y_train.min():.2f}")
print(f"  Max:  {y_train.max():.2f}")
print(f"  Mean: {y_train.mean():.2f}")
print(f"  Std:  {y_train.std():.2f}")

In [0]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Training Random Forest Model...")
print("="*80)

# Train model
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Make predictions
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print(f"✅ Random Forest Model Trained")
print(f"\nHyperparameters:")
print(f"  n_estimators: 100")
print(f"  max_depth: 20")
print(f"  n_features: {len(feature_cols)}")
print(f"\nTraining Metrics:")
print(f"  RMSE: {train_rmse:.4f} EUR/MWh")
print(f"  MAE:  {train_mae:.4f} EUR/MWh")
print(f"  R²:   {train_r2:.4f}")
print(f"\nTest Metrics:")
print(f"  RMSE: {test_rmse:.4f} EUR/MWh")
print(f"  MAE:  {test_mae:.4f} EUR/MWh")
print(f"  R²:   {test_r2:.4f}")

In [0]:
from sklearn.ensemble import GradientBoostingRegressor

print("Training Gradient Boosting Model...")
print("="*80)

# Train model (using sklearn's GradientBoosting as XGBoost alternative)
xgb_model = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    random_state=42
)

xgb_model.fit(X_train, y_train)

# Make predictions
y_pred_train_xgb = xgb_model.predict(X_train)
y_pred_test_xgb = xgb_model.predict(X_test)

# Calculate metrics
train_rmse_xgb = np.sqrt(mean_squared_error(y_train, y_pred_train_xgb))
test_rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_test_xgb))
train_mae_xgb = mean_absolute_error(y_train, y_pred_train_xgb)
test_mae_xgb = mean_absolute_error(y_test, y_pred_test_xgb)
train_r2_xgb = r2_score(y_train, y_pred_train_xgb)
test_r2_xgb = r2_score(y_test, y_pred_test_xgb)

print(f"✅ Gradient Boosting Model Trained")
print(f"\nHyperparameters:")
print(f"  n_estimators: 100")
print(f"  max_depth: 10")
print(f"  learning_rate: 0.1")
print(f"  n_features: {len(feature_cols)}")
print(f"\nTraining Metrics:")
print(f"  RMSE: {train_rmse_xgb:.4f} EUR/MWh")
print(f"  MAE:  {train_mae_xgb:.4f} EUR/MWh")
print(f"  R²:   {train_r2_xgb:.4f}")
print(f"\nTest Metrics:")
print(f"  RMSE: {test_rmse_xgb:.4f} EUR/MWh")
print(f"  MAE:  {test_mae_xgb:.4f} EUR/MWh")
print(f"  R²:   {test_r2_xgb:.4f}")

In [0]:
import matplotlib.pyplot as plt

# Determine which model performed better
if test_r2 > test_r2_xgb:
    best_model = rf_model
    best_model_name = "Random Forest"
    best_r2 = test_r2
else:
    best_model = xgb_model
    best_model_name = "Gradient Boosting"
    best_r2 = test_r2_xgb

print(f"Best Model: {best_model_name} (R² = {best_r2:.4f})")
print("\nExtracting Feature Importance...")
print("="*80)

# Get feature importance
importance = best_model.feature_importances_

# Create feature importance dataframe
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importance
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
for idx, row in importance_df.head(10).iterrows():
    print(f"  {row['feature']:40s} {row['importance']:.4f}")

# Visualize feature importance
plt.figure(figsize=(12, 8))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title(f'Feature Importance - {best_model_name}', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt

print("Creating Predictions Visualization...")
print("="*80)

# Get predictions from best model
if best_model_name == "Random Forest":
    y_pred = y_pred_test
else:
    y_pred = y_pred_test_xgb

# Create scatter plot: Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Scatter plot with perfect prediction line
axes[0].scatter(y_test, y_pred, alpha=0.5, s=20, edgecolors='k', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price (EUR/MWh)', fontsize=12)
axes[0].set_ylabel('Predicted Price (EUR/MWh)', fontsize=12)
axes[0].set_title(f'Actual vs Predicted Prices - {best_model_name}', 
                  fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Residuals
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, s=20, edgecolors='k', linewidth=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price (EUR/MWh)', fontsize=12)
axes[1].set_ylabel('Residuals (EUR/MWh)', fontsize=12)
axes[1].set_title('Residual Plot', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✅ Visualization complete")
print(f"\nResidual Statistics:")
print(f"  Mean:   {residuals.mean():.4f}")
print(f"  Std:    {residuals.std():.4f}")
print(f"  Min:    {residuals.min():.4f}")
print(f"  Max:    {residuals.max():.4f}")

In [0]:
# Model Comparison Summary
print("="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)

comparison_data = [
    ["Random Forest", train_rmse, test_rmse, train_mae, test_mae, train_r2, test_r2],
    ["Gradient Boosting", train_rmse_xgb, test_rmse_xgb, train_mae_xgb, test_mae_xgb, train_r2_xgb, test_r2_xgb]
]

comparison_df = pd.DataFrame(comparison_data, columns=[
    "Model", "Train RMSE", "Test RMSE", "Train MAE", "Test MAE", "Train R²", "Test R²"
])

print("\n")
display(comparison_df)

# Determine winner
print("\n" + "="*80)
print("WINNER DETERMINATION")
print("="*80)

if test_r2 > test_r2_xgb:
    winner = "Random Forest"
    winner_r2 = test_r2
    winner_rmse = test_rmse
    winner_mae = test_mae
else:
    winner = "Gradient Boosting"
    winner_r2 = test_r2_xgb
    winner_rmse = test_rmse_xgb
    winner_mae = test_mae_xgb

print(f"\n🏆 Best Model: {winner}")
print(f"\nPerformance Metrics:")
print(f"  Test R²:   {winner_r2:.4f} (explains {winner_r2*100:.2f}% of variance)")
print(f"  Test RMSE: {winner_rmse:.4f} EUR/MWh")
print(f"  Test MAE:  {winner_mae:.4f} EUR/MWh")

print("\n" + "="*80)
print("ML PIPELINE COMPLETE")
print("="*80)
print(f"\n✅ Both models trained successfully")
print(f"✅ {len(feature_cols)} features used")
print(f"✅ {len(X_train):,} training samples")
print(f"✅ {len(X_test):,} test samples")
print(f"\nModels: Random Forest vs Gradient Boosting")